# Chương 14. Từ mẫu dữ liệu đến bằng chứng

**Câu hỏi mở đầu:** Nếu kết luận dựa trên một mẫu, ta nên tin kết luận đó đến mức nào?

Notebook này là tài nguyên đồng hành của chương. Mỗi phần đều đi theo nhịp **câu hỏi → dữ liệu → mã → kết quả → diễn giải → kiểm tra bằng chứng**.

## Mục tiêu

- Tái hiện các ví dụ cốt lõi của chương bằng mã có thể chạy lại.
- Kiểm tra giả định trước khi diễn giải output.
- Kết thúc bằng ít nhất một câu hỏi về điều mà kết quả **chưa** cho biết.

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("error", category=FutureWarning)
warnings.filterwarnings("error", category=DeprecationWarning)
ROOT = Path.cwd()
DATA = ROOT / "data"
print("Working root:", ROOT)


In [ ]:
import pandas as pd
import numpy as np
pd.set_option("display.max_columns", 20)


In [ ]:
import numpy as np

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 100


## 1. Tổng thể, mẫu và ước lượng

In [ ]:
pop=pd.read_csv(DATA / "citypulse" / "citypulse_population.csv")
rand=pd.read_csv(DATA / "citypulse" / "citypulse_random_sample.csv")
web=pd.read_csv(DATA / "citypulse" / "citypulse_web_sample.csv")
def sat_rate(df): return df["satisfied"].eq("yes").mean()
print("population:",sat_rate(pop))
print("random sample:",sat_rate(rand))
print("web sample:",sat_rate(web))

## 2. Khoảng tin cậy xấp xỉ cho tỷ lệ

In [ ]:
def approx_ci(p_hat,n,z=1.96):
    se=np.sqrt(p_hat*(1-p_hat)/n)
    return p_hat-z*se, p_hat+z*se, se
for name,df in [("random",rand),("web",web)]:
    p=sat_rate(df)
    low,high,se=approx_ci(p,len(df))
    print(name,"p=",p,"SE=",se,"CI=",(low,high))

Mẫu web có khoảng khá hẹp nhưng ước lượng lệch xa tỷ lệ thật của tổng thể mô phỏng. Độ chụm cao không đảm bảo tính đại diện.

## 3. Bootstrap bằng NumPy

In [ ]:
rng=np.random.default_rng(20260827)
y=rand["satisfied"].eq("yes").to_numpy(dtype=float)
B=1200
boot=np.empty(B)
for b in range(B):
    boot[b]=rng.choice(y,size=len(y),replace=True).mean()
print("bootstrap mean:",boot.mean())
print("percentile CI:",np.quantile(boot,[.025,.975]))

## 4. So sánh tính đại diện

In [ ]:
for name,df in [("population",pop),("random",rand),("web",web)]:
    digital=df["digital_service_user"].eq("yes").mean()
    young=df["age_group"].eq("18-29").mean()
    print(name, "digital=", round(digital,3), "age18-29=", round(young,3))

### Kiểm tra bằng chứng

Khoảng tin cậy mô tả một nguồn độ không chắc chắn theo mô hình lấy mẫu; nó không sửa sai lệch chọn mẫu. Mẫu lớn hơn cũng không tự động đại diện hơn.

## Thực hành

Viết hai câu: một câu diễn giải đúng cho mẫu ngẫu nhiên và một câu giải thích vì sao mẫu web không nên được tin chỉ vì CI hẹp.

---
### Bạn đã sẵn sàng sang chương tiếp theo nếu có thể…

- giải thích output bằng lời;
- chỉ ra ít nhất một giả định;
- nói được kết quả chưa cho phép kết luận điều gì.

**Exit check:** Nếu mã chạy không lỗi nhưng câu trả lời trái với ý nghĩa của dữ liệu, bạn sẽ kiểm tra điều gì trước?